# Data Preprocessing Control Panel
## Choose your dataset, preprocessing method, and review results

This notebook provides a unified interface to:
1. Load your data
2. Choose a preprocessing method (basic, **Llama**, or **Gemini API**)
3. **Test with a small sample first** (NUM_SAMPLES = 5)
4. Review and compare results
5. Run on full dataset when satisfied

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Import the preprocessing module
from preprocessing import (
    load_data,
    basic_preprocess,
    preprocess_with_local_llama,  # For local Llama model
    preprocess_with_gemini_api,   # For Google Gemini API
    compare_cleaning_results
)

print("Preprocessing module loaded successfully")

✓ Preprocessing module loaded successfully


## Configuration

In [2]:
# ============================================
# DATA PATHS - Update these to match your setup
# ============================================

DATA_DIR = os.path.join(os.path.expanduser("~"), "Library", "CloudStorage", "Box-Box", "TMDATA", "dreamachine")

# Define your datasets
DATASETS = {
    'HS': {
        'input': 'freeform_HS_SensoryTool_complete.csv',
        'output_api': 'HS_reflections_APIcleaned.csv',
        'output_local': 'HS_reflections_LLAMAcleaned.csv'
    },
    'DL': {
        'input': 'freeform_DL_SensoryTool_complete.csv',
        'output_api': 'DL_reflections_APIcleaned.csv',
        'output_local': 'DL_reflections_LLAMAcleaned.csv'
    },
    'HW': {
        'input': 'freeform_HW_SensoryTool_complete.csv',
        'output_api': 'HW_reflections_APIcleaned.csv',
        'output_local': 'HW_reflections_LLAMAcleaned.csv'
    }
}

print(f"Data directory: {DATA_DIR}")
print(f"Available datasets: {', '.join(DATASETS.keys())}")

Data directory: /Users/rb666/Library/CloudStorage/Box-Box/TMDATA/dreamachine
Available datasets: HS, DL, HW


## Step 1: Select Dataset and Preprocessing Method

### WORKFLOW FOR COMPARING MODELS:
1. Set `NUM_SAMPLES = 5` (or 10) to test
2. Run each method on the same sample
3. Compare results in Step 4
4. Choose best method
5. Set `NUM_SAMPLES = None` to run on full data

In [3]:
# ============================================
# EDIT THESE 3 LINES
# ============================================

# Choose dataset: 'HS', 'DL', or 'HW'
DATASET_CHOICE = 'HS'

# Choose preprocessing method:
#   'basic'        → Sentence splitting, deduplication (no GPU/API needed)
#   'local_llama'  → Local Llama 3 model (needs GPU: Metal/CUDA)
#   'gemini_api'   → Google Gemini API (needs GOOGLE_API_KEY)
METHOD_CHOICE = 'basic'

# FOR TESTING: Set to a number (5, 10, etc) or None for full dataset
# TIP: Test with 5-10 samples first to compare methods, then set to None
NUM_SAMPLES = 5

print(f"\n{'='*60}")
print(f"CONFIGURATION")
print(f"{'='*60}")
print(f"Dataset:        {DATASET_CHOICE}")
print(f"Method:         {METHOD_CHOICE}")
print(f"Samples:        {NUM_SAMPLES if NUM_SAMPLES else 'ALL (FULL DATASET)'}")
print(f"{'='*60}")

if NUM_SAMPLES:
    print(f"\nRunning in TEST mode with {NUM_SAMPLES} samples")
    print(f"  (Good for checking if everything works before full run)")
else:
    print(f"\nRunning on FULL DATASET")
    print(f"  (This may take a while)")


CONFIGURATION
Dataset:        HS
Method:         basic
Samples:        5

Running in TEST mode with 5 samples
  (Good for checking if everything works before full run)


## Step 2: Load Data

In [4]:
# Build file paths
dataset_config = DATASETS[DATASET_CHOICE]
input_path = os.path.join(DATA_DIR, dataset_config['input'])

# Load data
df = load_data(input_path, text_column='reflection_answer', remove_na=True)

if df is not None:
    print(f"\nSuccessfully loaded {len(df)} records")
    print(f"DataFrame shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst 2 rows (preview):")
    display(df.head(2))
else:
    print("Failed to load data. Check file path and try again.")

Loaded 336 rows from freeform_HS_SensoryTool_complete.csv

Successfully loaded 336 records
DataFrame shape: (336, 8)
Columns: ['reflection_answer', 'emo_freeformResponse', 'aud_freeformResponse', 'vizCol_freeformAnswer', 'vizPat_freeformAnswer', 'vizMov_freeform_answer', 'cogBody_freeformAnswer', 'meta_HighSensory']

First 2 rows (preview):


,reflection_answer,emo_freeformResponse,aud_freeformResponse,vizCol_freeformAnswer,vizPat_freeformAnswer,vizMov_freeform_answer,cogBody_freeformAnswer,meta_HighSensory
39,Intense chaos. And then my mind checked out an...,NaN,NaN,NaN,NaN,NaN,NaN,1
44,good stuff.\n,NaN,often the visual aspect would change with a si...,NaN,NaN,NaN,NaN,1


## Step 3: Run Preprocessing

In [5]:
if df is None:
    print("No data loaded. Run Step 2 first.")
else:
    df_to_process = df.head(NUM_SAMPLES).copy() if NUM_SAMPLES else df.copy()
    
    # ============================================
    # BASIC METHOD: Sentence splitting + cleanup
    # ============================================
    if METHOD_CHOICE == 'basic':
        print("Running BASIC preprocessing...")
        texts = df_to_process['reflection_answer'].tolist()
        df_to_process = basic_preprocess(texts, split_into_sentences=True, min_words=2)
        print(f"✓ Basic preprocessing complete")
    
    # ============================================
    # LOCAL LLAMA METHOD (using Llama 3)
    # ============================================
    elif METHOD_CHOICE == 'local_llama':
        print("Running LOCAL LLAMA preprocessing...")
        print(f"(Using Llama 3 8B model with GPU acceleration)\n")
        
        output_path = os.path.join(DATA_DIR, dataset_config['output_local'])
        if NUM_SAMPLES:
            output_path = output_path.replace('.csv', f'_{NUM_SAMPLES}_test.csv')
        
        model_config = {
            'name': 'Llama-3-8B-Instruct',
            'repo': 'NousResearch/Meta-Llama-3-8B-Instruct-GGUF',
            'filename': 'Meta-Llama-3-8B-Instruct-Q4_K_M.gguf',
            'prompt_template': """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert data cleaner. Your task is to clean the user's text.
Follow these rules precisely:
1. Correct spelling mistakes and fix grammar.
2. Remove artifacts and formatting like '\\n'.
3. Do NOT change the original meaning or punctuation of the text.
4. Your response must contain ONLY the cleaned text, without any introductory phrases or commentary.<|eot_id|><|start_header_id|>user<|end_header_id|>
Clean the following text:

\"{text_to_clean}\"<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""
        }
        
        df_to_process = preprocess_with_local_llama(
            csv_path=input_path,
            output_path=output_path,
            model_config=model_config,
            text_column='reflection_answer',
            num_samples=NUM_SAMPLES
        )
        print(f"\n✓ Local Llama preprocessing complete")
    
    # ============================================
    # GEMINI API METHOD
    # ============================================
    elif METHOD_CHOICE == 'gemini_api':
        print("Running GEMINI API preprocessing...")
        print(f"(Using Google Gemini with batch processing)\n")
        
        output_path = os.path.join(DATA_DIR, dataset_config['output_api'])
        if NUM_SAMPLES:
            output_path = output_path.replace('.csv', f'_{NUM_SAMPLES}_test.csv')
        
        df_to_process = preprocess_with_gemini_api(
            csv_path=input_path,
            output_path=output_path,
            text_column='reflection_answer',
            batch_size=20,
            num_samples=NUM_SAMPLES
        )
        print(f"\n✓ Gemini API preprocessing complete")
    
    else:
        print(f"Unknown method: {METHOD_CHOICE}")
        print(f"Choose from: 'basic', 'local_llama', 'gemini_api'")

Running BASIC preprocessing...

Successfully loaded 8 texts.
After removing short texts (< 2 words), 7 remain.
After removing duplicates, 7 remain.
✓ Basic preprocessing complete


## Step 4: Review Results

### Summary

In [6]:
if df_to_process is not None and 'cleaned_reflection' in df_to_process.columns:
    print(f"\n{'='*80}")
    print(f"RESULTS SUMMARY")
    print(f"{'='*80}")
    print(f"Total processed: {len(df_to_process)}")
    print(f"Columns: {df_to_process.columns.tolist()}")
    print(f"\nDataFrame Info:")
    print(df_to_process.info())


RESULTS SUMMARY
Total processed: 7
Columns: ['reflection_answer', 'cleaned_reflection']

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   reflection_answer   7 non-null      object
 1   cleaned_reflection  7 non-null      object
dtypes: object(2)
memory usage: 244.0+ bytes
None


### View Full Results Table

In [7]:
# Show all rows for review
print(f"\nAll {len(df_to_process)} processed records:")
display(df_to_process)


All 7 processed records:


,reflection_answer,cleaned_reflection
0,Intense chaos.,Intense chaos.
1,And then my mind checked out and my subconscio...,And then my mind checked out and my subconscio...
2,What I imagine it's like looking back on life ...,What I imagine it's like looking back on life ...
3,good stuff.,good stuff.
4,a pattern of red and white lights thatflashed ...,a pattern of red and white lights thatflashed ...
5,i alsofelt sleep for abit and i was thinkingof...,i alsofelt sleep for abit and i was thinkingof...
6,i wentback to many hard and mostly beautiful m...,i wentback to many hard and mostly beautiful m...


### Side-by-Side Comparison (Original vs Cleaned)

In [8]:
if 'reflection_answer' in df_to_process.columns and 'cleaned_reflection' in df_to_process.columns:
    print(f"\n{'='*80}")
    print("ORIGINAL vs CLEANED COMPARISON")
    print(f"{'='*80}\n")
    
    for i in range(min(5, len(df_to_process))):
        print(f"[Record {i+1}]")
        print(f"ORIGINAL:")
        original = str(df_to_process['reflection_answer'].iloc[i])
        print(f"  {original[:200]}..." if len(original) > 200 else f"  {original}")
        print(f"\nCLEANED:")
        cleaned = str(df_to_process['cleaned_reflection'].iloc[i])
        print(f"  {cleaned[:200]}..." if len(cleaned) > 200 else f"  {cleaned}")
        print(f"{'-'*80}\n")


ORIGINAL vs CLEANED COMPARISON

[Record 1]
ORIGINAL:
  Intense chaos.

CLEANED:
  Intense chaos.
--------------------------------------------------------------------------------

[Record 2]
ORIGINAL:
  And then my mind checked out and my subconscious took over and started talking.

CLEANED:
  And then my mind checked out and my subconscious took over and started talking.
--------------------------------------------------------------------------------

[Record 3]
ORIGINAL:
  What I imagine it's like looking back on life before you die.

CLEANED:
  What I imagine it's like looking back on life before you die.
--------------------------------------------------------------------------------

[Record 4]
ORIGINAL:
  good stuff.

CLEANED:
  good stuff.
--------------------------------------------------------------------------------

[Record 5]
ORIGINAL:
  a pattern of red and white lights thatflashed andbecame more intensewhen the lights flashed intensely.

CLEANED:
  a pattern of red and wh

### Statistics

In [9]:
if 'reflection_answer' in df_to_process.columns and 'cleaned_reflection' in df_to_process.columns:
    print("\n" + "="*80)
    print("TEXT LENGTH STATISTICS")
    print("="*80 + "\n")
    
    original_lengths = df_to_process['reflection_answer'].astype(str).str.len()
    cleaned_lengths = df_to_process['cleaned_reflection'].astype(str).str.len()
    
    print(f"Original texts:")
    print(f"  Mean length: {original_lengths.mean():.0f} characters")
    print(f"  Min length: {original_lengths.min()} characters")
    print(f"  Max length: {original_lengths.max()} characters")
    
    print(f"\nCleaned texts:")
    print(f"  Mean length: {cleaned_lengths.mean():.0f} characters")
    print(f"  Min length: {cleaned_lengths.min()} characters")
    print(f"  Max length: {cleaned_lengths.max()} characters")
    
    print(f"\nDifference (cleaned - original):")
    diff = (cleaned_lengths - original_lengths)
    print(f"  Mean: {diff.mean():.0f} characters")
    print(f"  Min: {diff.min()} characters")
    print(f"  Max: {diff.max()} characters")


TEXT LENGTH STATISTICS

Original texts:
  Mean length: 61 characters
  Min length: 11 characters
  Max length: 102 characters

Cleaned texts:
  Mean length: 61 characters
  Min length: 11 characters
  Max length: 102 characters

Difference (cleaned - original):
  Mean: 0 characters
  Min: 0 characters
  Max: 0 characters


## Step 5: Run Full Dataset (When Satisfied)

After testing with `NUM_SAMPLES = 5` and reviewing results above:

1. **If results look good**: Uncomment code below
2. **Change `NUM_SAMPLES = 5` to `NUM_SAMPLES = None`**
3. **Re-run from Step 1 through Step 3** to process full dataset

In [11]:
# ============================================
# TO RUN FULL DATASET:
# ============================================
# 1. Go back to Step 1
# 2. Change: NUM_SAMPLES = 5  →  NUM_SAMPLES = None
# 3. Run Steps 1-3 again
# 4. Review results in Step 4
# ============================================

print("\nTo run the full dataset:")
print("  1. Go to Step 1 (Select Dataset and Method)")
print("  2. Change NUM_SAMPLES = 5 to NUM_SAMPLES = None")
print("  3. Run cells in Steps 1 through 3 again")
print("  4. Review final results in Step 4")


To run the full dataset:
  1. Go to Step 1 (Select Dataset and Method)
  2. Change NUM_SAMPLES = 5 to NUM_SAMPLES = None
  3. Run cells in Steps 1 through 3 again
  4. Review final results in Step 4
